# Lab 6: Working with Protein Families & Genome Assembly

**For this lab we go all in with command lines**

Part I: Working with protein families using HMMER (which we didn't finish last week)

Part II: Genome assembly using a small microbial genome

**Learning Goals**
- Understand how profile HMMs are built and used for sensitive protein family annotation
- Run `hmmscan` to annotate unknown proteins against an HMM database
- Understand the genome assembly workflow from raw reads to a finished assembly
- Perform quality control on sequencing reads with FastQC and Trimmomatic
- Assemble a small bacterial genome using SPAdes
- Evaluate assembly quality using QUAST and (optionally) BUSCO
- Understand key assembly statistics: N50, contig count, total length


## Part I: Working with Protein Families Using HMMER

HMMER is a suite of programs for biological sequence analysis using profile hidden Markov models (profile HMMs). Unlike BLAST, which searches sequences against sequences, HMMER leverages statistical models built from curated multiple sequence alignments (MSAs) to achieve more sensitive detection of distantly related homologs.

### Core Programs in the HMMER Suite

| Program | Input | Use Case |
|---|---|---|
| `hmmbuild` | MSA → profile HMM | Build a profile HMM from a curated alignment |
| `hmmsearch` | profile HMM vs sequence DB | Find all members of a family in a database |
| `hmmscan` | sequence vs HMM DB (e.g., Pfam) | Annotate domains in a query protein |
| `phmmer` | single sequence vs sequence DB | Protein–protein BLAST equivalent |
| `jackhmmer` | iterative single sequence | PSI-BLAST equivalent, iterative enrichment |
| `hmmpress` | HMM database | Compress & index HMM library for fast `hmmscan` |
| `hmmfetch` | HMM database | Retrieve individual HMMs from a library |

All exercises for this part will be done using the command line.

For this practice, we will use a small database of protein family HMMs and annotate query proteins by searching them against the family database.

If you have the active Bio2 environment created for this class, HMMER is ready for you to use.

> **NOTE:** Activate the Bio2 environment in terminal before running command lines:
> ```bash
> conda activate Bio2
> ```

See the [HMMER user guide](http://hmmer.org/documentation.html) for full documentation.

### Datasets

Available under `/nobackup/yye/Bio2/datasets/hmmer` (on silo):
- `RVT-All.hmm` — a database of HMMs of reverse transcriptases (RTs)
- `q1_protein.faa` — a FASTA file with 5 query proteins

**Copy these two files to your own working folder** (you cannot write to `/nobackup/yye/..`):
```bash
cp /nobackup/yye/Bio2/datasets/hmmer/RVT-All.hmm ~/lab06/
cp /nobackup/yye/Bio2/datasets/hmmer/q1_protein.faa ~/lab06/
cd ~/lab06/
```

### Step 1: Inspect the HMM Database

Before pressing the database, take a look at the raw HMM file:

```bash
head -40 RVT-All.hmm
```

You'll see records that begin with `HMMER3/f` and contain metadata like `NAME`, `LENG` (model length in columns), and `ACC` (accession). Each model encodes the consensus of a curated protein family alignment.

Count how many HMM models are in the file:
```bash
grep -c "^//" RVT-All.hmm
```

Each `//` marks the end of one HMM record.

### Step 2: Compress and Index the HMM Database

`hmmpress` prepares an HMM library for fast `hmmscan` lookups by creating binary index files:

```bash
hmmpress RVT-All.hmm
```

Inspect the files that were created:
```bash
ls -lh RVT-All.hmm*
```

You should see four new files:
- `.h3m` — binary HMM file
- `.h3i` — SSI index for fast model retrieval
- `.h3f` — optimized profile file
- `.h3p` — profiles in text format

> **Why do we need `hmmpress`?** `hmmscan` needs these binary index files for efficient lookup. Without them, searching would require reading the entire HMM text file for every query.

### Step 3: Inspect the Query Proteins

Look at the query sequences:
```bash
grep ">" q1_protein.faa
```

Count the number of sequences:
```bash
grep -c ">" q1_protein.faa
```

You can also view one full sequence:
```bash
head -20 q1_protein.faa
```

### Step 4: Search Query Proteins Against the HMM Database

Use `hmmscan` to annotate each query protein with matching RT family profiles:

```bash
hmmscan \
    --cpu 2 \
    -E 0.001 \
    --domE 0.001 \
    -o q1_vs_RVT.txt \
    --tblout q1_vs_RVT.tbl \
    --domtblout q1_vs_RVT.domtbl \
    RVT-All.hmm \
    q1_protein.faa
```

**What each flag means:**

| Flag | Meaning |
|---|---|
| `--cpu 2` | Use 2 CPU threads |
| `-E 0.001` | Report sequence-level hits with E-value ≤ 0.001 |
| `--domE 0.001` | Report domain-level hits with E-value ≤ 0.001 |
| `-o q1_vs_RVT.txt` | Full human-readable output |
| `--tblout q1_vs_RVT.tbl` | Per-sequence tabular output |
| `--domtblout q1_vs_RVT.domtbl` | Per-domain tabular output |

After the run finishes, inspect each output:
```bash
# Human-readable full output
less q1_vs_RVT.txt

# Tabular sequence-level hits (easier to parse)
grep -v "^#" q1_vs_RVT.tbl | column -t | head -20

# Domain-level hits (shows which part of the query matched)
grep -v "^#" q1_vs_RVT.domtbl | column -t | head -20
```

### Question 1.1

Based on your results, does `q1_protein.faa` contain proteins that are putative reverse transcriptases (RTs)?

- Which proteins (by sequence ID) had significant hits?
- What RT family did they match, and at what E-value?
- Were there any proteins with no significant match? What might that mean?

*Write your answer here:*

### Question 1.2

Look at the `--domtblout` file. For a protein with a significant hit, note the `hmm from` and `hmm to` columns versus the `ali from` and `ali to` columns.

- Does the match cover the full length of the HMM model, or only a region?
- What does a partial match (short `hmm from–to` range relative to model length) suggest about the query sequence?

*Write your answer here:*

### Question 1.3

Compare the E-values in the sequence-level table (`q1_vs_RVT.tbl`) versus the domain-level table (`q1_vs_RVT.domtbl`). 

- Why can the domain-level E-value be different from the sequence-level E-value?
- In what scenario would you trust domain-level annotations more than sequence-level ones?

*Write your answer here:*

---

## Part II: Genome Assembly of a Small Microbial Genome

### Background

Genome assembly is the process of reconstructing a genome sequence from many short sequencing reads. Modern short-read platforms (Illumina) produce billions of reads of 100–300 bp, and the assembler's job is to find overlaps between reads and build longer contiguous sequences (contigs), then scaffold them into larger pieces.

**The general workflow is:**

```
Raw reads (FASTQ)
      ↓
Quality Control (FastQC)
      ↓
Read Trimming (Trimmomatic or fastp)
      ↓
Genome Assembly (SPAdes)
      ↓
Assembly QC (QUAST)
      ↓
Completeness Check (BUSCO) [optional]
```

### Why Assemble a Microbial Genome?

Microbial genomes are small (typically 1–10 Mb), making them ideal for learning assembly concepts. A well-assembled bacterial genome enables:
- Gene prediction and functional annotation
- Comparative genomics across strains/species
- Identification of pathogenicity islands, antibiotic resistance genes, mobile elements
- Phylogenomic placement

### Dataset for This Lab

We will assemble a real paired-end Illumina dataset from a small bacterial genome, specially, a Staphylococcus aureus to demonstate the steps. 

This is a classic teaching genome: ~2.9 Mb, well-annotated, assembles from Illumina short reads, and finishes quickly.

The reads are from NCBI SRA accession SRR7706354 -- downsampled to have roughly 30x reads. 

We have downloaded these datasets and made them available under `/nobackup/yye/Bio2/datasets/assembly/` (on silo):
- `reads_R1.fastq` — forward reads 
- `reads_R2.fastq` — reverse reads 
- `reference.fna` — reference genome for comparison (note reference.fasta is also an assembly, not complete reference)
(if you are curious, you can check prepare-sml.sh under /nobackup/yye/Bio2/datasets/assembly/ to see where to download these datasets. )

Copy these to your working directory:
```bash
mkdir -p ~/lab06/assembly && cd ~/lab06/assembly
cp /nobackup/yye/Bio2/datasets/assembly/reads_R1.fastq .
cp /nobackup/yye/Bio2/datasets/assembly/reads_R2.fastq .
cp /nobackup/yye/Bio2/datasets/assembly/reference.fna .
```

### Step 2.1: Inspect the Raw Reads

Before doing anything else, look at the raw data. FASTQ files store each read as 4 lines:
1. Header line starting with `@`
2. DNA sequence
3. `+` separator
4. Quality scores (ASCII-encoded Phred scores)

```bash
# Look at the first few reads
cat reads_R1.fastq | head -20
```

Count the total number of reads:
```bash
# Each read = 4 lines, so divide line count by 4
cat reads_R1.fastq | wc -l | awk '{print $1/4, "reads"}'  
```

Check the read length:
```bash
cat reads_R1.fastq | awk 'NR==2{print length($0), "bp"; exit}'
```

Estimate coverage (reads × read_length / genome_size):
```bash
# Assuming ~4.5 Mb genome and 150 bp reads — adjust numbers as appropriate
# Coverage = (num_reads * 2 * read_length) / genome_size
# e.g.: (500000 * 2 * 150) / 4500000 ≈ 33x
echo "Estimated coverage calculation: (reads × 2 × read_length) / genome_size"
```

### Step 2.2: Quality Control with FastQC

FastQC generates a comprehensive HTML report of read quality metrics. Run it on both files simultaneously:

```bash
mkdir -p fastqc_raw
fastqc reads_R1.fastq reads_R2.fastq \
    --outdir fastqc_raw \
    --threads 2
```

Open the HTML reports:
```bash
ls fastqc_raw/
# Transfer the .html files to your local machine to view in a browser,
# or use a Jupyter-served HTML viewer if available
```

**Key sections to examine in the FastQC report:**

| Module | What to Look For |
|---|---|
| **Per base sequence quality** | Phred scores should be ≥ Q30 across most positions; quality often drops at the 3' end |
| **Per sequence quality scores** | Peak should be at Q30+ |
| **Per base sequence content** | Should be relatively flat; early bias is normal for some library preps |
| **Sequence duplication levels** | High duplication may indicate PCR over-amplification |
| **Adapter content** | If adapters are present (especially at 3' end), trimming is required |
| **Overrepresented sequences** | May indicate contamination or adapter dimers |

> **Phred Quality Score Reminder:**
> - Q20 = 99% accuracy (1 error per 100 bases)
> - Q30 = 99.9% accuracy (1 error per 1000 bases)
> - Q30 is generally the threshold for high-quality sequencing data

### Step 2.3: Trim Reads with Trimmomatic

Even high-quality datasets benefit from trimming adapter sequences and low-quality bases at read ends. Trimmomatic is a widely used tool for this purpose.

```bash
trimmomatic PE \
    reads_R1.fastq reads_R2.fastq \
    reads_R1_trimmed.fastq reads_R1_unpaired.fastq \
    reads_R2_trimmed.fastq reads_R2_unpaired.fastq \
    ILLUMINACLIP:TruSeq3-PE.fa:2:30:10 \
    LEADING:3 \
    TRAILING:3 \
    SLIDINGWINDOW:4:15 \
    MINLEN:36 \
    -threads 2
```

**What each Trimmomatic step does:**

| Step | Parameter | Effect |
|---|---|---|
| `ILLUMINACLIP` | `TruSeq3-PE.fa:2:30:10` | Remove Illumina TruSeq adapters; `2` mismatches allowed, `30` palindrome score, `10` simple clip score |
| `LEADING` | `3` | Remove bases from the start of a read with quality < 3 |
| `TRAILING` | `3` | Remove bases from the end of a read with quality < 3 |
| `SLIDINGWINDOW` | `4:15` | Scan with a 4-base sliding window; cut when average quality < 15 |
| `MINLEN` | `36` | Discard reads shorter than 36 bases after trimming |

After trimming, run FastQC again on trimmed reads to verify improvement:
```bash
mkdir -p fastqc_trimmed
fastqc reads_R1_trimmed.fastq reads_R2_trimmed.fastq \
    --outdir fastqc_trimmed \
    --threads 2
```

Compare the reports: adapter content should be gone, and per-base quality at the 3' end should be improved.

### Step 2.4: Assemble the Genome with SPAdes

SPAdes (St. Petersburg genome assembler) is one of the most widely used assemblers for bacterial genomes. It uses a de Bruijn graph approach, building graphs at multiple k-mer sizes and combining them.

#### What is a De Bruijn Graph?

A de Bruijn graph breaks each read into overlapping substrings of length *k* (k-mers). Nodes represent unique k-mers and edges represent overlaps. The assembler finds paths through this graph to reconstruct the original sequence.

```
Read:   ATCGATCG
k=4 kmers: ATCG → TCGA → CGAT → GATC → ATCG

Graph: ATCG → TCGA → CGAT → GATC → ATCG (loop = repeat)
```

Repeats longer than *k* cause branches in the graph — a major source of assembly fragmentation.

#### Run SPAdes

```bash
spades.py \
    -1 reads_R1_trimmed.fastq \
    -2 reads_R2_trimmed.fastq \
    -o spades_output \
    --careful \
    -t 4 \
    -m 16
```

**Key SPAdes flags:**

| Flag | Meaning |
|---|---|
| `-1` / `-2` | Forward and reverse paired-end reads |
| `-o` | Output directory |
| `--careful` | Reduces mismatches using BWA mapping (recommended for small genomes) |
| `-t 4` | Number of threads |
| `-m 16` | Memory limit in GB |

> SPAdes automatically selects a range of k-mer sizes (e.g., 21, 33, 55, 77) and combines the resulting graphs. You can see the k-mers it chose in `spades_output/params.txt`.

SPAdes will take several minutes. While it runs, you will see progress printed to the terminal.

#### Inspect SPAdes Output

```bash
ls spades_output/
```

Key output files:

| File | Description |
|---|---|
| `contigs.fasta` | Final assembled contigs (use for most analyses) |
| `scaffolds.fasta` | Scaffolded contigs (gaps filled with Ns using paired-end info) |
| `assembly_graph.fastg` | Assembly graph (can be visualized in Bandage) |
| `spades.log` | Full log with all parameters and progress |

Quick look at the contigs:
```bash
# Count contigs
grep -c ">" spades_output/contigs.fasta

# Look at the first contig header
grep ">" spades_output/contigs.fasta | head -10
```

SPAdes names contigs as: `>NODE_1_length_XXXX_cov_YY.Y`
- `length_XXXX` — contig length in bp
- `cov_YY.Y` — average k-mer coverage

### Step 2.5: Evaluate Assembly Quality with QUAST

QUAST (Quality Assessment Tool for Genome Assemblies) generates a detailed report of assembly statistics. If you have a reference genome, it can also compute reference-based metrics.

#### Without a Reference Genome

```bash
quast.py \
    spades_output/contigs.fasta \
    -o quast_output \
    --threads 2
```

#### With a Reference Genome (if available)

```bash
quast.py \
    spades_output/contigs.fasta \
    -r reference.fasta \
    -o quast_output_ref \
    --threads 2
```

View the summary report:
```bash
cat quast_output/report.txt
```

#### Understanding Key QUAST Statistics

| Metric | Definition | What it tells you |
|---|---|---|
| **# contigs** | Total number of assembled sequences | Fewer contigs = more complete assembly |
| **Total length** | Sum of all contig lengths (bp) | Should approximate the expected genome size |
| **Largest contig** | Length of the longest contig (bp) | Longer = better |
| **N50** | Length L such that 50% of total assembly length is in contigs ≥ L | Key summary statistic; higher is better |
| **L50** | Number of contigs needed to reach N50 | Lower is better |
| **GC (%)** | GC content | Should match known GC% for the organism |
| **# N's per 100 kbp** | Gap frequency (in scaffolds) | Lower is better |
| **Genome fraction (%)** | % of reference covered (reference-based only) | Higher is better |
| **Misassemblies** | Structural errors vs. reference (reference-based only) | Lower is better |

#### N50 Explained

Imagine sorting all contigs by length (longest first) and "filling up" the genome:

```
Contigs sorted by length:  [5000] [3000] [2000] [1500] [800] [500] ...
Cumulative sum:              5000   8000  10000  11500  12300 12800
Half of total (e.g. 6400):         ←-- N50 = 3000 (the contig that crosses the 50% mark)
```

A higher N50 means the assembly is in fewer, longer pieces — generally a sign of better contiguity.

### Step 2.6: Filter Contigs by Length (Optional but Recommended)

Short contigs (< 500 bp) are often assembly artifacts or from poorly covered regions. For downstream analyses, it is common to filter them out:

```bash
# Use awk to keep only contigs >= 500 bp
awk '/^>/{header=$0; seq=""; next} 
     {seq=seq$0} 
     END{print header"\n"seq}' spades_output/contigs.fasta | \
awk 'BEGIN{RS=">"; FS="\n"} NR>1 && length($2)>=500 {print ">"$0}' \
> contigs_filtered.fasta

# Or use seqkit (if available):
seqkit seq --min-len 500 spades_output/contigs.fasta > contigs_filtered.fasta

# Check how many contigs remain
grep -c ">" contigs_filtered.fasta
```

Re-run QUAST on the filtered file to compare:
```bash
quast.py contigs_filtered.fasta -o quast_filtered --threads 2
cat quast_filtered/report.txt
```

### Part II Questions

Answer the following questions based on your assembly results.

### Question 2.1 — Raw Data

Based on the `wc -l` count and read length you measured:
- How many paired-end reads are in the dataset (total, both R1 and R2)?
- What is the estimated sequencing coverage of the genome?
- Is this coverage sufficient for a reliable assembly? (Hint: 20–50x is typical for Illumina bacterial assemblies.)

*Write your answer here:*

### Question 2.2 — Quality Control

From the FastQC reports (before and after trimming):
- What was the overall quality of the raw reads (Phred score range)?
- Were adapters detected? If so, were they removed after trimming?
- Approximately how many reads were discarded by Trimmomatic? (Check Trimmomatic's stdout output.)
- Did trimming improve the per-base quality at the 3' end?

*Write your answer here:*

### Question 2.3 — Assembly Statistics

From your QUAST `report.txt`:
- How many contigs does your assembly have?
- What is the total assembly length? Does it match the expected genome size?
- What is the N50? What does this value tell you about assembly contiguity?
- What is the largest contig?

*Write your answer here:*

### Question 2.4 — Interpreting the Assembly

Look at the contig headers from SPAdes output (`grep ">" spades_output/contigs.fasta | head -20`).

- What is the coverage (`cov_`) of the longest contig? The shortest?
- Are there contigs with very low coverage (e.g., <5x)? What might these represent?
- After filtering to ≥500 bp, what fraction of contigs were removed? What fraction of total bases was lost?

*Write your answer here:*

### Question 2.5 — Factors Affecting Assembly Quality

Assembly quality depends on many factors. For each factor below, explain how it affects the assembly:

1. **Read depth (coverage):** How does very low coverage (< 5x) vs. very high coverage (> 200x) affect the result?
2. **Read length:** How would 250 bp reads compare to 75 bp reads for the same genome?
3. **Repetitive regions:** Why do repeats cause fragmentation in assemblies? What technologies can help resolve repeats?
4. **Genome size:** Why are microbial genomes much easier to assemble than plant or animal genomes?

*Write your answer here:*

---

## Summary: Full Workflow Command Reference

Here is the complete sequence of commands for both parts of this lab, for quick reference.

### Part I — HMMER

```bash
cd ~/lab06/

# Inspect HMM database
head -40 RVT-All.hmm
grep -c "^//" RVT-All.hmm

# Press (index) the HMM database
hmmpress RVT-All.hmm

# Inspect query proteins
grep ">" q1_protein.faa
grep -c ">" q1_protein.faa

# Search queries against HMM database
hmmscan --cpu 2 -E 0.001 --domE 0.001 \
    -o q1_vs_RVT.txt \
    --tblout q1_vs_RVT.tbl \
    --domtblout q1_vs_RVT.domtbl \
    RVT-All.hmm q1_protein.faa

# Inspect results
grep -v "^#" q1_vs_RVT.tbl | column -t | head -20
grep -v "^#" q1_vs_RVT.domtbl | column -t | head -20
```

### Part II — Genome Assembly

```bash
cd ~/lab06/assembly/

# 1. Inspect raw reads
zcat reads_R1.fastq.gz | head -8
zcat reads_R1.fastq.gz | wc -l | awk '{print $1/4, "reads"}'

# 2. Quality control
mkdir -p fastqc_raw
fastqc reads_R1.fastq.gz reads_R2.fastq.gz --outdir fastqc_raw --threads 2

# 3. Trim reads
trimmomatic PE \
    reads_R1.fastq.gz reads_R2.fastq.gz \
    reads_R1_trimmed.fastq.gz reads_R1_unpaired.fastq.gz \
    reads_R2_trimmed.fastq.gz reads_R2_unpaired.fastq.gz \
    ILLUMINACLIP:TruSeq3-PE.fa:2:30:10 LEADING:3 TRAILING:3 \
    SLIDINGWINDOW:4:15 MINLEN:36 -threads 2

mkdir -p fastqc_trimmed
fastqc reads_R1_trimmed.fastq.gz reads_R2_trimmed.fastq.gz \
    --outdir fastqc_trimmed --threads 2

# 4. Assemble with SPAdes
spades.py \
    -1 reads_R1_trimmed.fastq.gz \
    -2 reads_R2_trimmed.fastq.gz \
    -o spades_output \
    --careful -t 4 -m 16

grep -c ">" spades_output/contigs.fasta
grep ">" spades_output/contigs.fasta | head -10

# 5. Evaluate with QUAST
quast.py spades_output/contigs.fasta -o quast_output --threads 2
# With reference:
# quast.py spades_output/contigs.fasta -r reference.fasta -o quast_ref --threads 2
cat quast_output/report.txt

# 6. Filter short contigs
seqkit seq --min-len 500 spades_output/contigs.fasta > contigs_filtered.fasta
grep -c ">" contigs_filtered.fasta
quast.py contigs_filtered.fasta -o quast_filtered --threads 2

# 7. BUSCO (optional)
busco -i spades_output/contigs.fasta -o busco_output \
    -l bacteria_odb10 -m genome -c 4
cat busco_output/short_summary*.txt
```

## Further Reading

- [HMMER User Guide](http://hmmer.org/documentation.html)
- [SPAdes Manual](https://ablab.github.io/spades/)
- [QUAST Paper (Gurevich et al. 2013)](https://doi.org/10.1093/bioinformatics/btt086)
- [FastQC Documentation](https://www.bioinformatics.babraham.ac.uk/projects/fastqc/)
- [Trimmomatic Paper (Bolger et al. 2014)](https://doi.org/10.1093/bioinformatics/btu170)
- [A review of de Bruijn graph assembly (Compeau et al. 2011, Nature Biotechnology)](https://doi.org/10.1038/nbt.2023)